# 81 — Từ finlens tới Cerebro

Notebook đầu Track 8. Track 3 backtest kiểu **vectorised** — cả chuỗi giá là một
ma trận, tín hiệu và lợi suất là phép dịch cột. Track này dùng
[backtrader](https://www.backtrader.com/), một bộ máy **hướng sự kiện**: đi từng
thanh, chiến lược đặt lệnh, broker giả lập khớp lệnh, trừ phí, ghi sổ.

Câu hỏi của cả track: **finlens đưa dữ liệu gì, backtrader cần dữ liệu dạng gì,
và chỗ nối giữa hai bên sai âm thầm ở đâu.** Notebook này dựng chỗ nối đó bằng
tay, chạy một chiến lược đầu tiên, rồi đối chiếu một giao dịch tới từng đồng.

| Cạm bẫy | Triệu chứng | Đo bằng cách |
|---|---|---|
| **Giá kVND nạp thẳng** vào broker tính bằng VND | không có lỗi nào; số cổ phiếu gấp 1.000 lần | chạy bản sai và bản đúng cạnh nhau |
| **Phiên dở dang** khi gọi trong giờ giao dịch | thanh cuối trông bình thường, chỉ khối lượng còn thiếu và giá đóng cửa còn chạy | so khối lượng thanh cuối với các phiên trước |

In [1]:
import datetime as dt
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import backtrader as bt
import pandas as pd

import finlens
from finlens_examples import ap_dung_theme, duong, hom_nay, lui_ngay
from finlens_examples.charts import GIAM, TANG

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)
print(f"backtrader {bt.__version__} · pandas {pd.__version__} · finlens {finlens.__version__}")

backtrader 1.9.78.123 · pandas 3.0.5 · finlens 1.4.0


D:\finlens\finlens-python\finlens-python-example\venv\Lib\site-packages\anyio\_core\_tasks.py:275: UpgradeAvailableWarning: finlens 1.5.0 đã có (đang dùng 1.4.0). Nâng cấp: pip install -U finlens
  retval = await self._coro


## 1 · finlens đưa gì

Ba năm giá ngày của HPG. Trước khi đưa vào bất cứ đâu, đọc `df.attrs` — đó là
chỗ duy nhất frame nói nó đang tính bằng đơn vị gì.

In [2]:
MA = "HPG"
gia = client.eod.stock.ohlcv(MA, start=lui_ngay(HOM_NAY, nam=3))

meta = gia.attrs["finlens"]
print(f"{len(gia)} dòng · {gia['date'].min():%d/%m/%Y} → {gia['date'].max():%d/%m/%Y}")
print(f"Đơn vị: {meta['units']}")
print(f"Cơ sở giá: {meta['price_basis']}")
print()
print(gia.dtypes.to_string())
gia.tail(3)

746 dòng · 20/10/2023 → 20/10/2026
Đơn vị: {'symbol': None, 'date': None, 'open': 'kVND', 'high': 'kVND', 'low': 'kVND', 'close': 'kVND', 'volume': 'share'}
Cơ sở giá: adjusted

symbol            string
date      datetime64[ns]
open             float64
high             float64
low              float64
close            float64
volume           float64


,symbol,date,open,high,low,close,volume
743,HPG,2026-10-16,20.75,20.90,20.70,20.90,10189400.0
744,HPG,2026-10-19,21.00,21.05,20.55,20.55,13224600.0
745,HPG,2026-10-20,20.55,21.00,20.55,20.90,11382300.0


Bốn điều cần nhớ, vì cả bốn đều khác thứ backtrader mặc định:

1. **Giá tính bằng nghìn VND** (`kVND`) — `20.55` nghĩa là 20.550 đồng.
2. **Khối lượng tính bằng cổ phiếu**, không phải lô.
3. **`date` là một cột**, kiểu `datetime64` không múi giờ, chứ không phải index.
4. **Có cột `symbol`** kiểu `string` — frame finlens là dạng *long*, nhiều mã
   chung một bảng.

Giá mặc định là giá **điều chỉnh** (`price_basis = adjusted`) — đúng thứ cần cho
lợi suất. Notebook [`83`](83_luat_choi_viet_nam.ipynb) sẽ cho thấy khi nào phải
quay về giá thô.

## 2 · backtrader cần gì

`bt.feeds.PandasData` đọc một DataFrame theo **tham số ánh xạ cột**. Mỗi tham số
là tên trường của backtrader, giá trị là cách tìm cột tương ứng:

In [3]:
tham_so = dict(bt.feeds.PandasData.params._getitems())
pd.Series(
    {k: tham_so[k] for k in ["datetime", "open", "high", "low", "close", "volume", "openinterest"]},
    name="mặc định",
    dtype=object,
)

datetime        None
open              -1
high              -1
low               -1
close             -1
volume            -1
openinterest      -1
Name: mặc định, dtype: object

| Giá trị | Nghĩa |
|---|---|
| `None` ở `datetime` | lấy **index** làm thời gian |
| `-1` | tự tìm cột cùng tên (không phân biệt hoa thường) |
| `None` ở trường khác | trường này không có trong dữ liệu |

Từ đó ra ba việc phải làm với frame finlens: đặt `date` làm index, bỏ `symbol`
(một feed là một mã), và nói rõ `openinterest=None` — cổ phiếu không có hợp đồng
mở. Việc thứ tư — **đơn vị** — backtrader không biết và không hỏi. Nó nhận số
nào thì tính tiền bằng số đó.

Nhưng trước khi dựng feed, phải nhìn kỹ **thanh cuối cùng**.

## 3 · Cạm bẫy: phiên dở dang

finlens trả dữ liệu tới *hôm nay*. Nếu gọi trong giờ giao dịch, "hôm nay" là
một phiên **chưa đóng cửa**: giá `close` là giá khớp gần nhất, còn khối lượng
mới khớp được một phần.

Mốc "bây giờ" phải lấy từ đồng hồ **server** (+07), không phải đồng hồ máy bạn.

In [4]:
GIO_DONG_CUA = dt.time(15, 0)  # ATC kết thúc 14:45; chừa thời gian chốt dữ liệu

bay_gio = dt.datetime.fromisoformat(client.whoami()["server_time"])
thanh_cuoi = gia.iloc[-1]
la_hom_nay = thanh_cuoi["date"].date() == bay_gio.date()
dang_chay = la_hom_nay and bay_gio.time() < GIO_DONG_CUA

kl_truoc = gia["volume"].iloc[-2]
kl_trung_vi = gia["volume"].iloc[-21:-1].median()

print(f"Giờ server: {bay_gio:%d/%m/%Y %H:%M} (+07)")
print(f"Thanh cuối: {thanh_cuoi['date']:%d/%m/%Y} · close {thanh_cuoi['close']} · khối lượng {thanh_cuoi['volume']:,.0f} cp")
print(f"  = {thanh_cuoi['volume'] / kl_truoc:.1%} khối lượng phiên trước ({kl_truoc:,.0f} cp)")
print(f"  = {thanh_cuoi['volume'] / kl_trung_vi:.1%} trung vị 20 phiên ({kl_trung_vi:,.0f} cp)")
print()
print("⚠️ Phiên đang chạy — thanh cuối là phiên DỞ DANG" if dang_chay else "Thanh cuối là một phiên đã đóng cửa")

Giờ server: 20/10/2026 11:18 (+07)


Thanh cuối: 20/10/2026 · close 20.9 · khối lượng 11,382,300 cp
  = 86.1% khối lượng phiên trước (13,224,600 cp)
  = 75.5% trung vị 20 phiên (15,076,200 cp)

⚠️ Phiên đang chạy — thanh cuối là phiên DỞ DANG


Nếu dòng cuối báo **phiên dở dang**: thanh đó trông như mọi thanh khác — đủ năm
cột, không `NaN`, không cờ nào. Chỉ khối lượng tố cáo nó. Backtest trên thanh
này nghĩa là:

- **Tín hiệu tính trên một giá đóng cửa chưa tồn tại.** Giao cắt SMA "xảy ra"
  lúc 10 giờ sáng có thể biến mất lúc 14:45.
- **Đường vốn kết thúc ở một giá tạm.** Chạy lại buổi chiều ra con số khác,
  và không ai biết vì sao.
- **Chỉ báo dựa trên khối lượng** (OBV, khối lượng bất thường) đọc phiên này
  như một phiên cạn thanh khoản.

Cách sửa: bỏ thanh của hôm nay nếu server chưa qua giờ đóng cửa. Chạy sau 15:00
thì không có gì để bỏ.

In [5]:
hom_nay_ts = pd.Timestamp(bay_gio.date())
gia_sach = gia[gia["date"] < hom_nay_ts] if dang_chay else gia

print(f"Trước: {len(gia)} thanh, cuối {gia['date'].max():%d/%m/%Y}")
print(f"Sau:   {len(gia_sach)} thanh, cuối {gia_sach['date'].max():%d/%m/%Y}")

Trước: 746 thanh, cuối 20/10/2026
Sau:   745 thanh, cuối 19/10/2026


## 4 · Dựng feed bằng tay

Ba dòng, mỗi dòng sửa một chỗ lệch giữa hai thư viện.

In [6]:
COT_GIA = ["open", "high", "low", "close"]
HE_SO = {"kVND": 1_000, "VND": 1}[gia_sach.attrs["finlens"]["units"]["close"]]  # đọc, không đoán

khung = gia_sach.drop(columns="symbol").set_index("date")  # (1) một mã · (2) thời gian làm index
khung[COT_GIA] = khung[COT_GIA] * HE_SO  # (3) kVND → VND

print(f"Hệ số quy đổi đọc từ attrs: × {HE_SO:,}")
khung.tail(3)

Hệ số quy đổi đọc từ attrs: × 1,000


,open,high,low,close,volume
date,,,,,
2026-10-15,20600.0,21050.0,20600.0,20750.0,21160300.0
2026-10-16,20750.0,20900.0,20700.0,20900.0,10189400.0
2026-10-19,21000.0,21050.0,20550.0,20550.0,13224600.0


Mảnh thứ tư là `openinterest=None` khi tạo feed — nằm ngay trong lời gọi
`PandasData` ở mục dưới.

Lưu ý `HE_SO` **đọc từ `attrs`**, không viết cứng `* 1000`. Frame của chỉ số
(`VNINDEX`) tính bằng điểm, của phái sinh cũng vậy — nhân 1.000 vào đó cũng sai
âm thầm y như quên nhân với cổ phiếu. Bảng tra ở trên cố ý không có khoá cho
điểm chỉ số: gặp đơn vị lạ thì `KeyError`, không đoán.

## 5 · Cerebro và chiến lược đầu tiên

`Cerebro` là bộ điều phối: nhận feed, chiến lược, broker, rồi chạy vòng lặp.
Chiến lược đầu tiên là giao cắt SMA 20/50 — chọn vì nó đơn giản đến mức mọi giao
dịch đều kiểm được bằng tay, không phải vì nó tốt.

| Luật | Giá trị |
|---|---|
| Vào lệnh | SMA20 cắt **lên** SMA50, đang không giữ hàng |
| Thoát lệnh | SMA20 cắt **xuống** SMA50 |
| Khối lượng | 95% tiền mặt chia giá đóng cửa phiên tín hiệu, làm tròn xuống cổ phiếu |
| Phí | 0,15% mỗi chiều (`setcommission`) — thuế bán để notebook [`83`](83_luat_choi_viet_nam.ipynb) |
| Vốn | 100 triệu VND |

In [7]:
VON = 100_000_000  # VND
PHI = 0.0015  # 0,15% mỗi chiều


class GiaoCatSMA(bt.Strategy):
    """Mua khi SMA nhanh cắt lên SMA chậm, bán hết khi cắt xuống.

    ``co_dinh`` khác ``None`` thì mua đúng ngần ấy cổ phiếu thay vì theo tỷ lệ tiền.
    """

    params = (("nhanh", 20), ("cham", 50), ("ty_le", 0.95), ("co_dinh", None))

    def __init__(self):
        self.sma_nhanh = bt.ind.SMA(self.data.close, period=self.p.nhanh)
        self.sma_cham = bt.ind.SMA(self.data.close, period=self.p.cham)
        self.cat = bt.ind.CrossOver(self.sma_nhanh, self.sma_cham)
        self.nhat_ky = []

    def next(self):
        if not self.position and self.cat[0] > 0:
            so_cp = self.p.co_dinh or int(self.broker.getcash() * self.p.ty_le / self.data.close[0])
            self.buy(size=so_cp)
        elif self.position and self.cat[0] < 0:
            self.close()

    def notify_order(self, order):
        if order.status == order.Completed:
            self.nhat_ky.append(
                {
                    "ngay_tin_hieu": bt.num2date(order.created.dt),
                    "ngay_khop": bt.num2date(order.executed.dt),
                    "chieu": "mua" if order.isbuy() else "bán",
                    "so_cp": order.executed.size,
                    "gia_khop": order.executed.price,
                    "phi": order.executed.comm,
                    "tien_mat_sau": self.broker.getcash(),
                }
            )


def chay(feed, von=VON, **tham_so):
    """Một feed, một chiến lược, một broker. Trả về (strategy, giá trị cuối kỳ)."""
    cerebro = bt.Cerebro(stdstats=False)
    cerebro.adddata(feed)
    cerebro.addstrategy(GiaoCatSMA, **tham_so)
    cerebro.broker.setcash(von)
    cerebro.broker.setcommission(commission=PHI)
    strat = cerebro.run()[0]
    return strat, cerebro.broker.getvalue()


strat, gia_tri_cuoi = chay(bt.feeds.PandasData(dataname=khung, openinterest=None, name=MA))
nhat_ky = pd.DataFrame(strat.nhat_ky)

print(f"Vốn đầu {VON:,.0f} VND → cuối kỳ {gia_tri_cuoi:,.0f} VND ({gia_tri_cuoi / VON - 1:+.2%})")
print(f"{len(nhat_ky)} lệnh khớp")
nhat_ky.head(6)

Vốn đầu 100,000,000 VND → cuối kỳ 76,947,852 VND (-23.05%)
16 lệnh khớp


,ngay_tin_hieu,ngay_khop,chieu,so_cp,gia_khop,phi,tien_mat_sau
0,2024-05-22,2024-05-23,mua,4487,21460.0,144436.530,3.564543e+06
1,2024-07-16,2024-07-17,bán,-4487,21240.0,142955.820,9.872547e+07
2,2024-10-09,2024-10-10,mua,4584,20530.0,141164.280,4.474783e+06
3,2024-11-27,2024-11-28,bán,-4584,19680.0,135319.680,9.455258e+07
4,2024-12-17,2024-12-18,mua,4471,20090.0,134733.585,4.595460e+06
5,2025-01-10,2025-01-13,bán,-4471,19010.0,127490.565,8.946168e+07


In [8]:
sma = pd.DataFrame(
    {
        "date": khung.index,
        "Giá đóng cửa": khung["close"] / 1_000,
        "SMA20": khung["close"].rolling(20).mean() / 1_000,
        "SMA50": khung["close"].rolling(50).mean() / 1_000,
    }
).melt(id_vars="date", var_name="duong", value_name="gia")

fig = duong(
    sma,
    x="date",
    y="gia",
    theo="duong",
    tieu_de=f"{MA} — giao cắt SMA 20/50",
    phu_de="Tam giác là ngày KHỚP lệnh (phiên sau tín hiệu), tại giá mở cửa",
    nhan_y="nghìn VND",
)
for chieu, mau, ky_hieu in [("mua", TANG, "triangle-up"), ("bán", GIAM, "triangle-down")]:
    phan = nhat_ky[nhat_ky["chieu"] == chieu]
    fig.add_scatter(
        x=phan["ngay_khop"],
        y=phan["gia_khop"] / 1_000,
        mode="markers",
        name=chieu,
        marker=dict(color=mau, size=11, symbol=ky_hieu),
    )
fig

## 6 · Đối chiếu một giao dịch bằng tay

Một backtest chưa kiểm tay là một con số chưa ai đọc. Lấy giao dịch mua đầu tiên
và tính lại mọi thứ bằng pandas, không qua backtrader:

1. **Tín hiệu** — SMA20 phiên trước ≤ SMA50, phiên tín hiệu SMA20 > SMA50.
2. **Số cổ phiếu** — `int(95% × tiền mặt ÷ giá đóng cửa phiên tín hiệu)`.
3. **Giá khớp** — giá **mở cửa phiên sau**. Lệnh thị trường đặt sau khi phiên tín
   hiệu đóng cửa thì sớm nhất khớp được ở phiên kế tiếp.
4. **Tiền** — tiền mặt còn lại = vốn − giá trị khớp − phí.

In [9]:
mua = nhat_ky[nhat_ky["chieu"] == "mua"].iloc[0]
ban = nhat_ky[nhat_ky["chieu"] == "bán"].iloc[0]

sma20 = khung["close"].rolling(20).mean()
sma50 = khung["close"].rolling(50).mean()
i = khung.index.get_loc(pd.Timestamp(mua["ngay_tin_hieu"]))
print(
    f"Phiên trước tín hiệu {khung.index[i - 1]:%d/%m/%Y}: SMA20 {sma20.iloc[i - 1]:,.1f} "
    f"{'≤' if sma20.iloc[i - 1] <= sma50.iloc[i - 1] else '>'} SMA50 {sma50.iloc[i - 1]:,.1f}"
)
print(
    f"Phiên tín hiệu       {khung.index[i]:%d/%m/%Y}: SMA20 {sma20.iloc[i]:,.1f} "
    f"{'>' if sma20.iloc[i] > sma50.iloc[i] else '≤'} SMA50 {sma50.iloc[i]:,.1f}"
)
print(f"Phiên khớp           {khung.index[i + 1]:%d/%m/%Y}  ← phiên kế tiếp trong dữ liệu")

so_cp_tay = int(VON * 0.95 / khung["close"].iloc[i])
gia_tay = khung["open"].iloc[i + 1]
phi_tay = so_cp_tay * gia_tay * PHI
tien_tay = VON - so_cp_tay * gia_tay - phi_tay

doi_chieu = pd.DataFrame(
    {
        "backtrader": [mua["so_cp"], mua["gia_khop"], mua["phi"], mua["tien_mat_sau"]],
        "tính tay": [so_cp_tay, gia_tay, phi_tay, tien_tay],
    },
    index=["số cổ phiếu", "giá khớp (VND)", "phí (VND)", "tiền mặt sau lệnh (VND)"],
)
doi_chieu["lệch"] = doi_chieu["backtrader"] - doi_chieu["tính tay"]
doi_chieu.style.format("{:,.2f}")

Phiên trước tín hiệu 21/05/2024: SMA20 20,149.5 ≤ SMA50 20,211.8
Phiên tín hiệu       22/05/2024: SMA20 20,268.0 > SMA50 20,215.8
Phiên khớp           23/05/2024  ← phiên kế tiếp trong dữ liệu


,backtrader,tính tay,lệch
số cổ phiếu,"4,487.00","4,487.00",0.00
giá khớp (VND),"21,460.00","21,460.00",0.00
phí (VND),"144,436.53","144,436.53",0.00
tiền mặt sau lệnh (VND),"3,564,543.47","3,564,543.47",0.00


Lệch bằng 0 ở cả bốn dòng. Giờ đi nốt **vòng mua-bán** đó: lãi/lỗ ròng phải
bằng chênh giá nhân số cổ phiếu, trừ phí hai chiều.

In [10]:
j = khung.index.get_loc(pd.Timestamp(ban["ngay_tin_hieu"]))
gia_ban_tay = khung["open"].iloc[j + 1]
lai_lo_tay = mua["so_cp"] * (gia_ban_tay - mua["gia_khop"]) - mua["phi"] - mua["so_cp"] * gia_ban_tay * PHI
lai_lo_bt = ban["tien_mat_sau"] - VON

print(f"Mua {mua['so_cp']:,.0f} cp ngày {mua['ngay_khop']:%d/%m/%Y} giá {mua['gia_khop']:,.0f}")
print(f"Bán ngày {ban['ngay_khop']:%d/%m/%Y} giá {ban['gia_khop']:,.0f} (tay: mở cửa {khung.index[j + 1]:%d/%m/%Y} = {gia_ban_tay:,.0f})")
print(f"Lãi/lỗ ròng — backtrader: {lai_lo_bt:,.2f} VND · tính tay: {lai_lo_tay:,.2f} VND · lệch {lai_lo_bt - lai_lo_tay:,.2f}")

Mua 4,487 cp ngày 23/05/2024 giá 21,460
Bán ngày 17/07/2024 giá 21,240 (tay: mở cửa 17/07/2024 = 21,240)
Lãi/lỗ ròng — backtrader: -1,274,532.35 VND · tính tay: -1,274,532.35 VND · lệch 0.00


## 7 · Cạm bẫy: nạp giá kVND thẳng vào broker

Giờ làm sai đúng một chỗ: **bỏ dòng nhân 1.000**. Vốn vẫn là 100 triệu VND,
nhưng giá là `20.55` thay vì `20.550`. Không dòng nào khác thay đổi.

In [11]:
khung_kvnd = gia_sach.drop(columns="symbol").set_index("date")  # quên quy đổi

strat_sai, gia_tri_sai = chay(bt.feeds.PandasData(dataname=khung_kvnd, openinterest=None, name=MA))
nk_sai = pd.DataFrame(strat_sai.nhat_ky)
mua_sai = nk_sai[nk_sai["chieu"] == "mua"].iloc[0]
kl_phien_khop = khung.loc[pd.Timestamp(mua["ngay_khop"]), "volume"]

dong = [
    ("số cp lần mua đầu", mua["so_cp"], mua_sai["so_cp"], "{:,.0f}"),
    ("giá khớp", mua["gia_khop"], mua_sai["gia_khop"], "{:,.2f}"),
    ("tỷ trọng trong khối lượng phiên khớp", mua["so_cp"] / kl_phien_khop, mua_sai["so_cp"] / kl_phien_khop, "{:.3%}"),
    ("giá trị cuối kỳ (VND)", gia_tri_cuoi, gia_tri_sai, "{:,.0f}"),
    ("lợi suất cả kỳ", gia_tri_cuoi / VON - 1, gia_tri_sai / VON - 1, "{:+.2%}"),
]
bay = pd.DataFrame(
    {
        "đúng (VND)": [f.format(d) for _, d, _, f in dong],
        "sai (kVND)": [f.format(s) for _, _, s, f in dong],
        "sai ÷ đúng": [f"{s / d:,.4f}" for _, d, s, _ in dong],
    },
    index=[t for t, *_ in dong],
)
print(f"Lỗi / cảnh báo từ backtrader: không có. {len(nk_sai)} lệnh khớp, y như bản đúng ({len(nhat_ky)}).")
bay

Lỗi / cảnh báo từ backtrader: không có. 16 lệnh khớp, y như bản đúng (16).


,đúng (VND),sai (kVND),sai ÷ đúng
số cp lần mua đầu,"4,487","4,487,482","1,000.1074"
giá khớp,"21,460.00",21.46,0.0010
tỷ trọng trong khối lượng phiên khớp,0.011%,11.374%,"1,000.1074"
giá trị cuối kỳ (VND),"76,947,852","76,943,837",0.9999
lợi suất cả kỳ,-23.05%,-23.06%,1.0002


Đọc bảng từ dưới lên, vì đó là thứ tự người ta thường đọc kết quả backtest:

- **Lợi suất cả kỳ gần như trùng nhau.** Số cổ phiếu gấp 1.000, giá nhỏ đi
  1.000 — tích của chúng vẫn là số tiền đó. Phí tính theo phần trăm nên cũng
  không đổi. Chênh lệch còn lại chỉ do làm tròn số cổ phiếu. Đây là lý do lỗi
  này sống sót: **thước đo người ta nhìn đầu tiên không hề động đậy.**
- **Số cổ phiếu gấp 1.000 lần** (làm tròn ở hai thang khác nhau nên lệch khoảng
  một phần vạn), và chiếm **khoảng 11% khối lượng của cả phiên khớp** thay vì
  0,01%. Một tài khoản 100 triệu đồng không thể mua một phần chín cả phiên HPG.

Lỗi sẽ không vô hại mãi. Nó nổ ra ngay khi chiến lược có một thứ gì **tính bằng
cổ phiếu hay bằng đồng tuyệt đối** — lô 100, bước giá 10/50/100 đồng, phí cố định
mỗi lệnh, giới hạn tỷ trọng khối lượng, hay đơn giản là một lệnh cố định 1.000 cổ
phiếu:

In [12]:
_, cuoi_dung = chay(bt.feeds.PandasData(dataname=khung, openinterest=None), co_dinh=1_000)
_, cuoi_sai = chay(bt.feeds.PandasData(dataname=khung_kvnd, openinterest=None), co_dinh=1_000)

print("Cùng chiến lược, mỗi lần mua cố định 1.000 cổ phiếu:")
print(f"  đúng (VND):  lãi/lỗ {cuoi_dung - VON:>+16,.2f} VND")
print(f"  sai  (kVND): lãi/lỗ {cuoi_sai - VON:>+16,.2f} VND")
print(f"  tỷ lệ: {(cuoi_dung - VON) / (cuoi_sai - VON):,.0f} lần")

Cùng chiến lược, mỗi lần mua cố định 1.000 cổ phiếu:
  đúng (VND):  lãi/lỗ    -5,624,520.00 VND
  sai  (kVND): lãi/lỗ        -5,624.52 VND
  tỷ lệ: 1,000 lần


Lãi/lỗ của bản sai nhỏ đi **đúng 1.000 lần** — mọi khoản lãi, lỗ và phí đều
co lại cùng một tỷ lệ, và trên một tài khoản 100 triệu thì nó trông như một
chiến lược gần hoà vốn. Vẫn không có lỗi nào. Cách phòng duy nhất: **quy về VND
ngay ở bước tạo feed, đọc hệ số từ `attrs`**, và từ đó trở đi mọi con số trong
backtrader đọc thẳng ra đồng.

## 8 · Gói lại: `bt_vn.feed_tu_finlens`

Mọi bước vừa dựng tay — bỏ phiên dở dang, đặt index, bỏ `symbol`, đọc đơn vị từ
`attrs`, nhân hệ số, tắt `openinterest` — đã được gom vào
[`finlens_examples/bt_vn.py`](../../finlens_examples/bt_vn.py) để các notebook sau
dùng lại. Không có gì bị giấu: kiểm nó cho đúng kết quả của bản tay.

In [13]:
from finlens_examples import bt_vn

gia_module = bt_vn.bo_phien_chua_dong(gia, client)
khung_module = bt_vn.chuan_bi_frame(gia_module)

print(f"bo_phien_chua_dong: {len(gia_module)} thanh · bản tay: {len(gia_sach)} thanh")
pd.testing.assert_frame_equal(khung_module, khung, check_names=False)
print("chuan_bi_frame: khung giống hệt bản tay (giá trị, cột, index)")

strat_m, gia_tri_m = chay(bt_vn.feed_tu_finlens(gia_module))
nk_m = pd.DataFrame(strat_m.nhat_ky)
pd.testing.assert_frame_equal(nk_m, nhat_ky)
print(f"Nhật ký lệnh: {len(nk_m)} lệnh, trùng từng ô với bản tay")
print(f"Giá trị cuối kỳ: module {gia_tri_m:,.2f} · bản tay {gia_tri_cuoi:,.2f} · lệch {gia_tri_m - gia_tri_cuoi:,.2f} VND")

bo_phien_chua_dong: 745 thanh · bản tay: 745 thanh
chuan_bi_frame: khung giống hệt bản tay (giá trị, cột, index)


Nhật ký lệnh: 16 lệnh, trùng từng ô với bản tay
Giá trị cuối kỳ: module 76,947,852.17 · bản tay 76,947,852.17 · lệch 0.00 VND


Module còn làm thêm một việc bản tay không làm: **từ chối đoán đơn vị**. Ghép
hai frame lấy từ hai lời gọi khác nhau thì `attrs` hai bên khác nhau (khác
`request_id`, `fetched_at`), nên pandas bỏ luôn — và frame ghép không còn nói nó
tính bằng gì.

In [14]:
moc = lui_ngay(HOM_NAY, nam=1)
nua_dau = client.eod.stock.ohlcv(MA, start=lui_ngay(HOM_NAY, nam=3), end=moc)
nua_sau = client.eod.stock.ohlcv(MA, start=moc)
ghep = pd.concat([nua_dau, nua_sau]).drop_duplicates(subset="date")
print(f"attrs sau khi ghép: {ghep.attrs}")

try:
    bt_vn.feed_tu_finlens(ghep)
except ValueError as loi:
    print(f"ValueError: {loi}")

feed_ghep = bt_vn.feed_tu_finlens(ghep, don_vi_gia="kVND")  # nói rõ thì mới chạy
print("Truyền don_vi_gia='kVND' → tạo feed được")

attrs sau khi ghép: {}
ValueError: Frame không còn df.attrs['finlens'] (thường do pd.concat/merge). Truyền don_vi_gia='kVND' | 'VND' | 'index_point' cho rõ.
Truyền don_vi_gia='kVND' → tạo feed được


Bản tay ở mục 4 sẽ ném `KeyError` ở `attrs["finlens"]` trong tình huống này —
cũng là dừng lại, nhưng với thông báo khó hiểu. Còn một đoạn code viết cứng
`* 1000` thì chạy tiếp, và đúng cho tới ngày ai đó đưa vào một frame chỉ số.

## Tổng kết

| Việc | Bản tay | `bt_vn` |
|---|---|---|
| Bỏ phiên dở dang | so `date` cuối với giờ server, trước 15:00 thì bỏ | `bo_phien_chua_dong(df, client)` |
| Thời gian làm index, bỏ `symbol` | `drop(columns="symbol").set_index("date")` | `chuan_bi_frame(df)` |
| Quy về VND | hệ số đọc từ `attrs["finlens"]["units"]` | như bên trái; mất `attrs` thì **dừng** |
| Feed | `PandasData(dataname=..., openinterest=None)` | `feed_tu_finlens(df)` |
| Kiểm tra | đối chiếu một giao dịch tới từng đồng | — luôn tự làm |

Hai điều mang theo sang các notebook sau:

1. **Lệnh thị trường khớp ở giá mở cửa phiên sau tín hiệu.** Điều này đúng, và
   notebook kế tiếp sẽ cho thấy một tuỳ chọn của backtrader phá nó như thế nào.
2. **Lỗi đơn vị không làm hỏng lợi suất — nó làm hỏng số cổ phiếu.** Nên nó nổ ra
   ở đúng chỗ luật chơi Việt Nam bắt đầu: lô 100, bước giá, biên độ.

---

**Tiếp theo:** [`82_giai_phau_strategy.ipynb`](82_giai_phau_strategy.ipynb) —
`lines`, `next`, bốn loại lệnh, và lệnh khớp ở đâu.